# Time Series Models

This chapter will focus on markov processes, a subset of the more general stochastic processes that are used to model time series.

## Representations

### N-order Markov Chains

- AR
- MA
- ARIMA
- SARIMA
- GARCH
- VAR

### Hidden Markov Models

https://ericmjl.github.io/essays-on-data-science/machine-learning/markov-models/

Variations:
- GMM - HMM https://stats.stackexchange.com/questions/171307/how-to-train-a-gaussian-mixture-hidden-markov-model

In [1]:
import argparse
import logging
import sys

import torch
import torch.nn as nn
from torch.distributions import constraints

import pyro
import pyro.contrib.examples.polyphonic_data_loader as poly
import pyro.distributions as dist
from pyro import poutine
from pyro.infer.autoguide import AutoDelta
from pyro.infer import SVI, JitTraceEnum_ELBO, TraceEnum_ELBO, TraceTMC_ELBO
from pyro.ops.indexing import Vindex
from pyro.optim import Adam
from pyro.util import ignore_jit_warnings

logging.basicConfig(format='%(relativeCreated) 9d %(message)s', level=logging.DEBUG)

# Add another handler for logging debugging events (e.g. for profiling)
# in a separate stream that can be captured.
log = logging.getLogger()
debug_handler = logging.StreamHandler(sys.stdout)
debug_handler.setLevel(logging.DEBUG)
debug_handler.addFilter(filter=lambda record: record.levelno <= logging.DEBUG)
log.addHandler(debug_handler)

In [2]:
data = poly.load_data(poly.JSB_CHORALES)

processing raw data - jsb_chorales ...
dumped processed data to /opt/anaconda3/envs/ml/lib/python3.7/site-packages/pyro/contrib/examples/.data/jsb_chorales.pkl


In [19]:
sequences = data['train']['sequences']
lengths = data['train']['sequence_lengths']

# find all the notes that are present at least once in the training set
present_notes = ((sequences == 1).sum(0).sum(0) > 0)

In [23]:
num_observations = float(lengths.sum())

In [24]:
num_observations

13807.0

## Inference Algorithms

Problem Space:

1. Discrete Time + Scalar Discrete Latent / State Space + No assumption on transmission / emission probabilities
- Filtering: Forward Algorithm
- Smoothing: Backward Algorithm
- Predicting: 

2. Discrete Time + Vector Continuous Latent / State Space + Linear Gaussian transmission / emission probabilities
- Filtering: Kalman Filter
- Smoothing: RTS Smoother
- Predicting: 

3. Discrete Time + Vector Continuous Latent / State Space + Locally-linear Gaussian transmission / emission probabilities
- Filtering: Extended Kalman Filter
- Smoothing: RTS Smoother
- Predicting: 

4. Discrete Time + Vector Continuous Latent / State Space + Non-linear Gaussian transmission / emission probabilities
- Filtering: Unscented Kalman Filter
- Smoothing: RTS Smoother
- Predicting: 

5. Discrete Time + Vector Continuous Latent / State Space + Non-linear Non-Gaussian transmission / emission probabilities
- Filtering: Particle Filters
- Smoothing: RTS Smoother
- Predicting: 

Continuous Time Markov Processes
Stochastic Processes
	- Gaussian Processes 
		- Weiner Process
		- Ornstein Uhlenbeck
		- Brownian Bridge 
	- Dirichlet Processes